# Đánh giá lại mô hình với checkpoint `results/checkpoint-3102`
Notebook này sẽ sử dụng checkpoint đã huấn luyện để đánh giá lại trên tập validation và test.

In [1]:
# Load checkpoint, tokenizer và chuẩn bị tập validation và test
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import os
checkpoint_path = '../results/checkpoint-1500'
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Hàm load dữ liệu val/test (giả sử đã có sẵn file json)
import json
def load_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

val_data = load_dataset('../data/val_dataset.json')
test_data = load_dataset('../data/test_dataset.json')

/home/vinh/anaconda3/envs/vit5-chatbot/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-23 18:33:31.248696: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-23 18:33:31.263193: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753270411.277844 2566267 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753270411.281802 2566267 cuda_blas.c

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Hàm sinh output và đánh giá BLEU/ROUGE/USR/CMADE cho tập val/test
import evaluate
from tqdm import tqdm
import numpy as np
from collections import Counter
import re

bleu = evaluate.load('sacrebleu')
rouge = evaluate.load('rouge')

def calculate_usr(predictions, references):
    """
    Calculate USR (Unigram Subsequence Recall) score
    USR measures the coverage of unigrams from reference in prediction
    """
    usr_scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens = set(pred.lower().split())
        ref_tokens = set(ref.lower().split())
        if len(ref_tokens) == 0:
            usr_scores.append(0.0)
        else:
            overlap = len(pred_tokens.intersection(ref_tokens))
            usr = overlap / len(ref_tokens)
            usr_scores.append(usr)
    return np.mean(usr_scores)

def calculate_cmade(predictions, references):
    """
    Calculate CMADE (Character-level Minimum Average Distance Edit) score
    CMADE measures character-level edit distance normalized by reference length
    """
    def edit_distance(s1, s2):
        """Calculate Levenshtein distance between two strings"""
        if len(s1) < len(s2):
            return edit_distance(s2, s1)
        
        if len(s2) == 0:
            return len(s1)
        
        previous_row = list(range(len(s2) + 1))
        for i, c1 in enumerate(s1):
            current_row = [i + 1]
            for j, c2 in enumerate(s2):
                insertions = previous_row[j + 1] + 1
                deletions = current_row[j] + 1
                substitutions = previous_row[j] + (c1 != c2)
                current_row.append(min(insertions, deletions, substitutions))
            previous_row = current_row
        
        return previous_row[-1]
    
    cmade_scores = []
    for pred, ref in zip(predictions, references):
        edit_dist = edit_distance(pred.lower(), ref.lower())
        if len(ref) == 0:
            cmade_scores.append(1.0 if len(pred) > 0 else 0.0)
        else:
            cmade = edit_dist / len(ref)
            cmade_scores.append(cmade)
    return np.mean(cmade_scores)

def generate_and_eval(dataset):
    preds, refs = [], []
    for item in tqdm(dataset):
        input_text = item['prompt'] if 'prompt' in item else item['prompt']
        ref = item['response'] if 'response' in item else item['response']
        inputs = tokenizer(input_text, return_tensors='pt', truncation=True, padding=True).to(device)
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=128)
        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        preds.append(pred)
        refs.append(ref)
    
    # Calculate all metrics
    bleu_score = bleu.compute(predictions=preds, references=[[r] for r in refs])['score']
    rouge_score = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    usr_score = calculate_usr(preds, refs)
    cmade_score = calculate_cmade(preds, refs)
    
    print(f"BLEU: {bleu_score:.2f}")
    # Handle both old and new rouge score formats
    if hasattr(rouge_score['rougeL'], 'mid'):
        print(f"ROUGE-L: {rouge_score['rougeL'].mid.fmeasure:.4f}")
    else:
        print(f"ROUGE-L: {rouge_score['rougeL']:.4f}")
    print(f"USR: {usr_score:.4f}")
    print(f"CMADE: {cmade_score:.4f}")
    
    return preds, refs

In [3]:
# Đánh giá trên tập validation
print('Đánh giá trên tập validation:')
val_preds, val_refs = generate_and_eval(val_data)

# Đánh giá trên tập test
print('Đánh giá trên tập test:')
test_preds, test_refs = generate_and_eval(test_data)

Đánh giá trên tập validation:


100%|██████████| 1206/1206 [07:32<00:00,  2.66it/s]


BLEU: 33.75
ROUGE-L: 0.5906
USR: 0.5455
CMADE: 0.5378
Đánh giá trên tập test:


100%|██████████| 594/594 [03:45<00:00,  2.63it/s]


BLEU: 33.60
ROUGE-L: 0.5966
USR: 0.5475
CMADE: 0.5504
